# Stenosis Detection — Kaggle (plug & play)

**YOLO11s @ 768px on ARCADE task-2 + Danilov (capped) + CADICA (capped).** Target **F1 > 0.57**, recall-first gate `{f1: 0.57, recall: 0.60}`.

## Do this first, then Run All
1. **Settings → Accelerator → GPU** (T4).
2. **Settings → Internet → ON** (pip + git clone).
3. **+ Add Input**: attach **ARCADE**, **danilov-stenosis** (title contains *danilov*), and **CADICA** (title contains *cadica*). CADICA adds patient diversity — the #1 accuracy lever. Any subset works; missing ones auto-skip.
4. **Run All.** First pass keeps `DRY_RUN = True` (3-epoch wiring check). Then set `DRY_RUN = False` and Run All again for the real **80-epoch** run (val saturated ~ep16 in the 150-epoch run; patience early-stops anyway).

Everything else auto-detects (dataset paths at any depth, OOM fallback to batch 8, output collection). No paths to edit.

**Phase 1 tuning baked in (see [`docs/STAGE2_PHASE1_POA.md`](../docs/STAGE2_PHASE1_POA.md)):** the config now carries a domain-tuned `augment:` block (mosaic **0.0**, scale 0.2, erasing 0.0, no HSV jitter, box 9.0/dfl 2.0, cos_lr) — YOLO's COCO defaults shrink/erase the tiny faint lesions. CADICA is **capped to 40 frames/patient** and its frames now group by patient (`pXX`) in the split. **§5b** adds per-source val, an operating-point sweep, and **temporal-voting per-video sensitivity** (the deployment metric) — all no-retrain, on the trained `best.pt`.

**Honesty + safety baked in:** a **leakage audit** (§3b) runs *before* training and **hard-fails** if any patient/clip lands in both train and val. Danilov is **capped to 5 frames/patient**. Training enforces the **recall-first gate** and eval runs at low conf so recall isn't throttled. After training, §5 writes a val-only GT-vs-pred demo to `outputs/stenosis_demo.mp4`.

## 1 · GPU + clone + install

In [ ]:
import os, sys, torch
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU.'
print('CUDA:', torch.cuda.get_device_name(0))

REPO_SLUG    = 'jugalmodi0111/interventional-imaging-pipeline'
GITHUB_TOKEN = ''                              # '' if repo is PUBLIC, else paste a PAT
REPO         = '/kaggle/tmp/interventional-imaging-pipeline'  # /kaggle/tmp NOT saved to output -> repo+9k PNGs stay out of download
if not os.path.exists(REPO):
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone -q https://{auth}github.com/{REPO_SLUG}.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install 'ultralytics>=8.2' pycocotools opencv-python pyyaml
from src import env
E = env.setup()
torch.backends.cudnn.benchmark = True

In [ ]:
# --- CADICA (optional, patient-diverse add) ---------------------------------------------------
# FASTEST: + Add Input a public CADICA Kaggle dataset (title contains 'cadica') -> this cell auto-skips.
#   e.g. 'abszgbert/invasive-coronary-angiography-cadica-dataset'  or  'ariadnapartinen/cadica-...'
# ELSE this pulls the 2.87GB zip straight from Mendeley over Kaggle's fast pipe (no local up/download).
import os, glob
DOWNLOAD_CADICA = True                                    # set False to skip CADICA entirely
CADICA_DL = ''
_attached = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
             if 'cadica' in os.path.basename(os.path.normpath(d)).lower()]
if _attached:
    print('CADICA already attached under /kaggle/input -> skipping download')
elif DOWNLOAD_CADICA:
    CADICA_URL = ('https://data.mendeley.com/public-files/datasets/p9bpx9ctcv/'
                  'files/9088fc7c-629b-4a23-8553-5a4d3a5282e1/file_downloaded')   # CADICA.zip (~2.87GB, v5)
    os.makedirs('/kaggle/tmp/cadica', exist_ok=True)
    if not glob.glob('/kaggle/tmp/cadica/**/selectedVideos', recursive=True):
        print('downloading CADICA (~2.87GB) from Mendeley ...')
        os.system(f'curl -L --retry 3 -o /kaggle/tmp/CADICA.zip "{CADICA_URL}"')
        os.system('unzip -q -o /kaggle/tmp/CADICA.zip -d /kaggle/tmp/cadica')
    hits = glob.glob('/kaggle/tmp/cadica/**/selectedVideos', recursive=True)
    CADICA_DL = os.path.dirname(hits[0]) if hits else '/kaggle/tmp/cadica'
    print('CADICA ready ->', CADICA_DL, '| groundtruth files:',
          len(glob.glob(CADICA_DL + '/**/groundtruth/*.txt', recursive=True)))
else:
    print('CADICA skipped (DOWNLOAD_CADICA=False, none attached)')

In [ ]:
import glob, os
# ARCADE: the folder that CONTAINS stenosis/*/annotations/*.json
ah = glob.glob('/kaggle/input/**/stenosis/*/annotations/*.json', recursive=True)
assert ah, 'No ARCADE stenosis json under /kaggle/input — + Add Input the ARCADE dataset.'
ARCADE_INPUT = ah[0].split('/stenosis/')[0]

def _find(word):                                          # any *word* dir at ANY depth (shallowest match)
    hits = [os.path.normpath(d) for d in glob.glob('/kaggle/input/**/', recursive=True)
            if word in os.path.basename(os.path.normpath(d)).lower()]
    return min(hits, key=len) if hits else ''

DANILOV_INPUT = _find('danilov')
CADICA_INPUT  = _find('cadica') or CADICA_DL              # attached mirror, else the Mendeley download above

print('inputs attached:', os.listdir('/kaggle/input'))
print('ARCADE_INPUT  =', ARCADE_INPUT)
print('DANILOV_INPUT =', DANILOV_INPUT or '(none)')
print('CADICA_INPUT  =', CADICA_INPUT or '(none)')

os.makedirs('data/raw', exist_ok=True)
os.system(f'ln -sfn "{ARCADE_INPUT}" data/raw/arcade')
if DANILOV_INPUT:
    os.system(f'ln -sfn "{DANILOV_INPUT}" data/raw/danilov')
if CADICA_INPUT:
    os.system(f'ln -sfn "{CADICA_INPUT}" data/raw/cadica')
assert glob.glob('data/raw/arcade/stenosis/**/*.json', recursive=True), 'ARCADE symlink wrong'
if DANILOV_INPUT:
    print('danilov bmp:', len(glob.glob('data/raw/danilov/**/*.bmp', recursive=True)),
          '| xml:', len(glob.glob('data/raw/danilov/**/*.xml', recursive=True)))
if CADICA_INPUT:
    print('cadica groundtruth files:',
          len(glob.glob('data/raw/cadica/**/groundtruth/*.txt', recursive=True)))

## 3 · Standardize ARCADE (+ Danilov) → YOLO

In [ ]:
import yaml, glob
cfg = yaml.safe_load(open('configs/stenosis_yolo.yaml'))
if not DANILOV_INPUT:
    cfg['datasets'].pop('danilov', None)                  # ARCADE-only if Danilov not attached
if not CADICA_INPUT:
    cfg['datasets'].pop('cadica', None)                   # skip CADICA if not attached
from src.data_prep import danilov_to_yolo, cadica_to_yolo
danilov_to_yolo.main(cfg)                                  # ARCADE (+ Danilov, capped 5/patient via max_frames_per_patient)
if CADICA_INPUT:
    print('cadica:', cadica_to_yolo.main(cfg), 'frames')   # patient-diverse add (keyframes only, patient-grouped split)
ntr = len(glob.glob('data/processed/stenosis/images/train/*'))
nva = len(glob.glob('data/processed/stenosis/images/val/*'))
print(f'train imgs: {ntr} | val imgs: {nva}')
assert ntr and nva, 'conversion produced no images — check the printed per-dataset counts above'

## 3b · Leakage audit (hard gate)
Verifies the split is **patient-grouped**, not per-frame — the check that makes the F1 trustworthy. Runs **before** training and **raises** (stops Run All) if either:
1. any patient/clip group appears in **both** train and val, or
2. the Danilov frames were **not** collapsed by `group_key` (their filenames don't match the expected `<site>_<patient>_<seq>_<frame>` pattern → silent per-frame leak).

If this cell raises, **the reported F1 would be inflated** — fix the split before trusting any number.

In [ ]:
# HARD GATE — no leak, no false F1. Build the TRUE Danilov stem set from raw (independent of the
# regex) so a silent grouping no-op is caught, then audit the produced YOLO split.
import os
from src.data_prep.io_utils import audit_split_leakage

_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
danilov_stems = None
if DANILOV_INPUT:
    danilov_stems = {os.path.splitext(f)[0]
                     for _, _, fs in os.walk('data/raw/danilov') for f in fs
                     if os.path.splitext(f)[1].lower() in _EXTS}

rep = audit_split_leakage('data/processed/stenosis', danilov_stems=danilov_stems)  # raises on any leak
print('LEAKAGE CHECK PASSED — split is patient-grouped and honest')
print(f"  train {rep['train_imgs']} imgs / {rep['train_groups']} groups | "
      f"val {rep['val_imgs']} imgs / {rep['val_groups']} groups "
      f"(val ~{rep['val_frac_by_group']:.0%} by group)")
if rep['danilov']:
    d = rep['danilov']
    print(f"  danilov: {d['danilov_frames']} frames -> {d['patient_groups']} patients, "
          f"{d['ungrouped']} ungrouped ({d['ungrouped_frac']:.0%})")

# Extra integrity checks (WARN — surface silent-but-bad training conditions):
from src.data_prep.io_utils import duplicate_basenames_across_cocos
dupes = duplicate_basenames_across_cocos('data/raw/arcade/stenosis')
if dupes:
    print(f'\n[info] ARCADE cross-split basename repeats: {len(dupes)} names in >1 split '
          f'-> auto-disambiguated by split prefix in coco_to_yolo (no data loss).')
    print('       e.g.', list(dupes)[:5])
vg = rep['val_groups']
if vg < 10:
    print(f'\n[WARN] only {vg} distinct patient/clip groups in val -> F1 will be noisy; '
          'treat the number as a wide estimate.')

## 4 · Train YOLO11s @ 768
Leave `DRY_RUN = True` for a 3-epoch wiring check. Set `False` for the real **80-epoch** run (target **F1 > 0.57**). OOM auto-falls back to batch 8.

The config now flows a domain-tuned **`augment:`** block through `train_kwargs` → `model.train()` (mosaic 0.0, scale 0.2, erasing 0.0, no HSV, box 9.0/dfl 2.0, cos_lr) — tuned for tiny faint grayscale lesions, replacing YOLO's COCO defaults. `train()` prints F1 (recall-weighted from the PR curve) and an explicit `[PASS]/[FAIL]` line against the **recall-first gate** `{f1: 0.57, recall: 0.60}`. SSL (pseudo-label + gdino) stays guarded inside `train()`: skipped unless a disjoint `ssl.unlabeled_dir` exists, so it can't re-leak val frames.

In [ ]:
import os, glob
DRY_RUN = False         # honest re-run: patient-grouped split (io_utils.group_key). Run All.

if DRY_RUN:
    cfg['train']['epochs'] = 3
    cfg['ssl'] = {'pseudo_label': False}                  # skip SSL on the fast pass

# SSL pseudo-labeling copies ALL ssl.unlabeled_dir frames into images/train. If that dir holds val
# patients' frames it re-leaks them past the audit. Only allow SSL when a disjoint, existing
# unlabeled dir is actually attached; otherwise disable it (default here -> no leak).
_ssl = cfg.get('ssl', {})
_udir = _ssl.get('unlabeled_dir')
_has_unlabeled = bool(_udir and os.path.isdir(_udir)
                      and glob.glob(os.path.join(_udir, '**', '*.png'), recursive=True))
# Both pseudo-label AND the gdino cold-start copy every unlabeled_dir frame into images/train.
# Without a disjoint unlabeled set that silently re-leaks val patients -> disable both.
if not _has_unlabeled and (_ssl.get('pseudo_label') or _ssl.get('seed') == 'gdino'):
    cfg['ssl'] = dict(_ssl, pseudo_label=False, seed=None)
    print('SSL disabled (pseudo-label + gdino seed): no disjoint ssl.unlabeled_dir attached '
          '-> would re-leak val frames into train. Attach an unlabeled-only dir (e.g. XCAD) to enable.')

from src.train.train_detector import train, train_kwargs, run_tag
TAG = run_tag(cfg)
print('run:', TAG, '| kwargs ->', train_kwargs(cfg))
try:
    best = train(cfg, project=f"{E['runs']}/stenosis/{TAG}")
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        torch.cuda.empty_cache(); cfg['train']['batch'] = 8
        print('CUDA OOM at batch 16 -> retrying at batch 8'); best = train(cfg, project=f"{E['runs']}/stenosis/{TAG}")
    else:
        raise
print('best weights ->', best)

## 5 · Demo video — GT vs prediction on held-out (val) patients
Runs `best.pt` over the **val** split (unseen patients) and writes an overlay mp4:
**green = model prediction (+conf), red = ground truth.** Saved to `outputs/stenosis_demo.mp4` and copied to `/kaggle/working` for download. Val-only → an honest look at generalization, not the (now impossible) leaked frames.

In [ ]:
import os, glob, shutil, cv2
from ultralytics import YOLO

VAL_IMG, VAL_LBL = 'data/processed/stenosis/images/val', 'data/processed/stenosis/labels/val'
frames = sorted(glob.glob(os.path.join(VAL_IMG, '*.png')))[:400]   # cap demo length
assert frames, 'no val frames to render'
os.makedirs('outputs', exist_ok=True)
DEMO = 'outputs/stenosis_demo.mp4'
model = YOLO(best)
h, w = cv2.imread(frames[0]).shape[:2]
vw = cv2.VideoWriter(DEMO, cv2.VideoWriter_fourcc(*'mp4v'), 12, (w, h))
for fp in frames:
    img = cv2.imread(fp)                                          # already CLAHE'd 768 png
    H, W = img.shape[:2]
    lp = os.path.join(VAL_LBL, os.path.splitext(os.path.basename(fp))[0] + '.txt')
    if os.path.exists(lp):                                        # ground truth -> red
        for ln in open(lp):
            p = ln.split()
            if len(p) == 5:
                cx, cy, bw, bh = (float(v) for v in p[1:])
                cv2.rectangle(img, (int((cx-bw/2)*W), int((cy-bh/2)*H)),
                              (int((cx+bw/2)*W), int((cy+bh/2)*H)), (0, 0, 200), 2)
    r = model.predict(fp, conf=0.25, imgsz=W, verbose=False)[0]   # prediction -> green
    for b in r.boxes:
        x1, y1, x2, y2 = (int(v) for v in b.xyxy[0].tolist())
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(img, f'{float(b.conf[0]):.2f}', (x1, max(12, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)
    cv2.putText(img, 'green=pred  red=GT  (val / unseen patients)', (8, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    vw.write(img)
vw.release()
if os.path.isdir('/kaggle/working'):
    shutil.copy(DEMO, '/kaggle/working/stenosis_demo.mp4')
print('demo video ->', DEMO, '| frames:', len(frames))

## 5b · Phase 1 evaluation — per-source · operating point · temporal voting
Quick-win diagnostics on the trained `best.pt` (**no retrain**). Each writes a `.txt` to `/kaggle/working` for download. See [`docs/STAGE2_PHASE1_POA.md`](../docs/STAGE2_PHASE1_POA.md).

- **P1.0 — per-source val:** ARCADE vs CADICA vs Danilov. Tells you *where* the 74% miss lives → aims Phase 2 spend.
- **P1.1a — operating-point sweep:** pick the recall-first confidence knee (a missed stenosis is the costly error). Does **not** lift the PR curve — chooses where to operate on it.
- **P1.1b — temporal voting:** per-video sensitivity over the **full raw CADICA cine**. A lesion persists across frames, so voting recovers per-frame misses and drops 1-frame flicker. **This is the deployment-relevant metric** — and it's CoreML-neutral (host-side post-processing). Compare `raw` vs `voted`.

In [ ]:
# §5b · P1.0 — per-source val (ARCADE / CADICA / Danilov). WHERE does recall fail? Aims Phase 2.
# Splits the merged val set by dataset of origin (inferred from stem naming) and runs .val() on each.
import io, contextlib, os
from src.eval import val_by_source

_buf = io.StringIO()
try:
    with contextlib.redirect_stdout(_buf):
        val_by_source.main(best, 'data/processed/stenosis')
except Exception as e:
    _buf.write(f'per-source val failed: {e}\n')
_out = 'per-source val (P=precision R=recall; low conf 0.001 so recall is not throttled)\n' + _buf.getvalue()
print(_out)
if os.path.isdir('/kaggle/working'):
    open('/kaggle/working/phase1_val_by_source.txt', 'w').write(_out)
    print('saved -> /kaggle/working/phase1_val_by_source.txt')

In [ ]:
# §5b · P1.1a — operating-point sweep: pick the recall-first confidence knee (no retrain).
# F1 peaks near conf 0.20, but a missed stenosis is the costly error -> for a screening flag pick a
# LOWER conf (more recall) where precision is still tolerable. This does not lift the PR curve; it
# chooses where on it to operate. Cross-check the chosen conf with P1.1b's temporal-voting cell.
from ultralytics import YOLO

DATA = 'data/processed/stenosis/data.yaml'
m_op = YOLO(best)
rows = ['conf   P      R      F1     mAP50']
for c in (0.05, 0.10, 0.15, 0.20, 0.25):
    b = m_op.val(data=DATA, conf=c, verbose=False).box
    p, r = float(b.mp), float(b.mr); f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    rows.append(f'{c:.2f}   {p:.3f}  {r:.3f}  {f1:.3f}  {float(b.map50):.3f}')
report = ('operating-point sweep (recall-first: pick the knee where R rises but P still acceptable)\n'
          + '\n'.join(rows))
print(report)
if os.path.isdir('/kaggle/working'):
    open('/kaggle/working/phase1_operating_point.txt', 'w').write(report)
    print('\nsaved -> /kaggle/working/phase1_operating_point.txt')

In [ ]:
# §5b · P1.1b — temporal-voting per-video sensitivity over the FULL raw CADICA cine.
# The processed val set is CADICA KEYFRAMES ONLY (annotation-driven). A stenosis persists across the
# cine, so we run the detector on EVERY input frame of each val-patient video, temporally vote
# (recover detector-missed frames by interpolation, drop 1-frame flicker), then score recovery on the
# annotated keyframes. This per-video sensitivity is the deployment-relevant metric — a screening flag
# fires if the lesion is caught ANYWHERE in the clip, not on every single frame.
import os, glob
from src.serve.temporal_vote import aggregate_sequence, iou_xywhn

VOTE_CONF = 0.10     # recall-first inference conf (cross-check with the P1.1a sweep above)
IOU_HIT   = 0.3      # a prediction "hits" a GT box at IoU >= this
MIN_HITS  = 2        # temporal_vote persistence knob (2 = drop 1-frame flicker, keep corroborated)

_tv_lines = []
def _tv_log(s):
    print(s); _tv_lines.append(s)

if not os.path.isdir('data/raw/cadica'):
    _tv_log('CADICA raw not attached -> skipping temporal-voting eval (P1.1b).')
else:
    import cv2
    from ultralytics import YOLO
    from src.data_prep.cadica_to_yolo import cadica_patient_of, parse_cadica_gt, _sibling_dir
    from src.data_prep.io_utils import split_of, clahe_unsharp
    m_tv = YOLO(best)
    IMGSZ = int((cfg.get('model') or {}).get('imgsz', 768))

    # CADICA video dirs whose PATIENT is in the val split (honest: unseen patients only).
    vids = []
    for input_dir in sorted(glob.glob('data/raw/cadica/**/input', recursive=True)):
        patient = cadica_patient_of(input_dir)
        if patient and split_of(patient) == 'val':
            vids.append((patient, os.path.dirname(input_dir), input_dir))
    _tv_log(f'CADICA val videos: {len(vids)} (patients hashed to the val split)')

    def _norm_gt(boxes, W, H):
        return [((x + w / 2) / W, (y + h / 2) / H, w / W, h / H) for (x, y, w, h) in boxes]
    def _hit(preds, gts):
        return any(iou_xywhn(pb, gb) >= IOU_HIT for pb in preds for gb in gts)

    pos = raw_det = voted_det = 0
    for patient, video_dir, input_dir in vids:
        gt_dir = _sibling_dir(video_dir, 'groundtruth')
        frames = sorted(glob.glob(os.path.join(input_dir, '*')),
                        key=lambda p: os.path.splitext(os.path.basename(p))[0])
        seq, key_gt = [], {}     # seq: ordered per-frame dets; key_gt[i]: GT boxes on annotated frames
        for i, fp in enumerate(frames):
            g = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
            if g is None:
                seq.append([]); continue
            H, W = g.shape
            frame = cv2.resize(clahe_unsharp(g), (IMGSZ, IMGSZ))   # match train/inference preprocessing
            r = m_tv.predict(frame, conf=VOTE_CONF, imgsz=IMGSZ, verbose=False)[0].boxes
            seq.append([{'box': tuple(bb), 'conf': float(cc)}
                        for bb, cc in zip(r.xywhn.tolist(), r.conf.tolist())])
            stem = os.path.splitext(os.path.basename(fp))[0]
            gp = os.path.join(gt_dir, stem + '.txt') if gt_dir else ''
            if gp and os.path.isfile(gp):
                gt = _norm_gt(parse_cadica_gt(open(gp).read()), W, H)
                if gt:
                    key_gt[i] = gt
        if not key_gt:
            continue
        pos += 1
        raw_ok = any(_hit([d['box'] for d in seq[i]], gts) for i, gts in key_gt.items())
        stab = aggregate_sequence(seq, iou_thr=IOU_HIT, min_hits=MIN_HITS, conf_agg='mean')
        vote_ok = any(_hit([d['box'] for d in stab[i]], gts) for i, gts in key_gt.items())
        raw_det += int(raw_ok); voted_det += int(vote_ok)

    _tv_log(f'\nper-video sensitivity (lesion videos, conf {VOTE_CONF}, min_hits {MIN_HITS}, IoU {IOU_HIT})')
    _tv_log(f'  lesion videos scored: {pos}')
    if pos:
        _tv_log(f'  raw   per-video sensitivity: {raw_det}/{pos} = {raw_det / pos:.3f}')
        _tv_log(f'  voted per-video sensitivity: {voted_det}/{pos} = {voted_det / pos:.3f}  (temporal-voting lift)')

if os.path.isdir('/kaggle/working'):
    open('/kaggle/working/phase1_temporal_voting.txt', 'w').write('\n'.join(_tv_lines))
    print('\nsaved -> /kaggle/working/phase1_temporal_voting.txt')

## 6 · Collect outputs for download
Copies `best.pt` + `results.csv` to `/kaggle/working` and zips the full run. Grab them from the **Output** panel (right).

In [ ]:
import shutil, os
run_dir = os.path.dirname(os.path.dirname(best))          # .../<TAG>/base
for rel in ('weights/best.pt', 'results.csv'):
    src = os.path.join(run_dir, rel)
    if os.path.exists(src):
        dst = f'/kaggle/working/{os.path.basename(rel)}'
        shutil.copy(src, dst); print('download ->', dst)
shutil.make_archive('/kaggle/working/stenosis_run', 'zip', run_dir)
print('full run zip -> /kaggle/working/stenosis_run.zip')
print('\nNext (on your Mac): python -m src.export.yolo_to_coreml --weights best.pt')